In [2]:
import pandas as pd
import numpy as np

from sklearn.datasets import load_wine
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import VotingClassifier, BaggingClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.cluster import KMeans
from lightgbm import LGBMClassifier

import mglearn
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sb
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

In [3]:
wine = load_wine()

wine_df = pd.DataFrame(wine.data, columns=wine.feature_names)
wine_df['target'] = wine.target

print(wine_df.head())

   alcohol  malic_acid   ash  alcalinity_of_ash  magnesium  total_phenols  \
0    14.23        1.71  2.43               15.6      127.0           2.80   
1    13.20        1.78  2.14               11.2      100.0           2.65   
2    13.16        2.36  2.67               18.6      101.0           2.80   
3    14.37        1.95  2.50               16.8      113.0           3.85   
4    13.24        2.59  2.87               21.0      118.0           2.80   

   flavanoids  nonflavanoid_phenols  proanthocyanins  color_intensity   hue  \
0        3.06                  0.28             2.29             5.64  1.04   
1        2.76                  0.26             1.28             4.38  1.05   
2        3.24                  0.30             2.81             5.68  1.03   
3        3.49                  0.24             2.18             7.80  0.86   
4        2.69                  0.39             1.82             4.32  1.04   

   od280/od315_of_diluted_wines  proline  target  
0          

```text
와인데이터셋 이용
- train, test 나누기
- 정규화 하기
- 앙상블 이용하여 train,test (정확도,f1 score , confusion matrix)확인하기
- 부스팅 모델을 이용한 train,test (정확도,f1 score , confusion matrix)확인하기

iris 데이터 셋 이용
- 군집화 하기
- 정답과 군집화값 정확도 확인하기 

In [4]:
x_data = wine.data
y_data = wine.target

In [5]:
x_data

array([[1.423e+01, 1.710e+00, 2.430e+00, ..., 1.040e+00, 3.920e+00,
        1.065e+03],
       [1.320e+01, 1.780e+00, 2.140e+00, ..., 1.050e+00, 3.400e+00,
        1.050e+03],
       [1.316e+01, 2.360e+00, 2.670e+00, ..., 1.030e+00, 3.170e+00,
        1.185e+03],
       ...,
       [1.327e+01, 4.280e+00, 2.260e+00, ..., 5.900e-01, 1.560e+00,
        8.350e+02],
       [1.317e+01, 2.590e+00, 2.370e+00, ..., 6.000e-01, 1.620e+00,
        8.400e+02],
       [1.413e+01, 4.100e+00, 2.740e+00, ..., 6.100e-01, 1.600e+00,
        5.600e+02]], shape=(178, 13))

In [6]:
y_data

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2])

In [7]:
scaleF = StandardScaler()
x_scale = scaleF.fit_transform(x_data)

In [8]:
# train, test 나누기
# 정규화 하기
logistic = LogisticRegression(max_iter=500)
knn = KNeighborsClassifier(n_neighbors=5)
tree = DecisionTreeClassifier(min_samples_leaf=5) # 과적합이 많이 발생함으로 min_samples_leaf 을 통해 조절

voting_model = VotingClassifier([('logistic',logistic),('knn', knn),('tree',tree)],
                                voting='soft') #soft : 확률 평균

x_train, x_test, y_train, y_test = train_test_split(x_scale, y_data, test_size=0.2,
                                                    random_state=42, stratify=y_data)

voting_model.fit( x_train, y_train )

,estimators,"[('logistic', ...), ('knn', ...), ...]"
,voting,'soft'
,weights,None
,n_jobs,None
,flatten_transform,True
,verbose,False
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True


앙상블 이용하여 train,test (정확도,f1 score , confusion matrix)확인하기

train

In [9]:
pred_t = voting_model.predict(x_train)
pred_t

array([0, 0, 0, 0, 2, 2, 1, 2, 0, 0, 1, 1, 0, 0, 2, 1, 1, 1, 0, 0, 1, 2,
       0, 2, 2, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 2, 0, 1, 0, 1, 1, 1, 1, 1,
       1, 1, 0, 0, 2, 0, 0, 2, 1, 1, 2, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0,
       0, 1, 1, 1, 2, 2, 0, 0, 2, 1, 1, 1, 1, 2, 0, 2, 0, 2, 1, 2, 2, 1,
       0, 0, 1, 1, 1, 0, 0, 0, 2, 1, 1, 1, 2, 1, 0, 1, 0, 1, 2, 0, 0, 1,
       2, 2, 1, 1, 1, 2, 2, 2, 2, 1, 1, 2, 1, 2, 2, 0, 0, 2, 0, 1, 2, 2,
       1, 2, 1, 1, 1, 2, 1, 0, 2, 2])

In [10]:
voting_model.score(x_train, y_train)

1.0

In [11]:
f1_t = f1_score(y_train, pred_t, average='macro')
f1_t

1.0

In [12]:
confusion_matrix(y_train, pred_t)

array([[47,  0,  0],
       [ 0, 57,  0],
       [ 0,  0, 38]])

test

In [13]:
pred_tt = voting_model.predict(x_test)
pred_tt

array([0, 2, 0, 1, 1, 0, 0, 1, 1, 2, 1, 2, 0, 2, 0, 1, 1, 0, 1, 0, 1, 1,
       0, 0, 1, 1, 0, 2, 1, 2, 0, 2, 1, 2, 2, 2])

In [14]:
voting_model.score(x_train,y_train)

1.0

In [15]:
voting_model.score(x_test,y_test)

1.0

In [16]:
f1_tt = f1_score(y_test, pred_tt, average='macro')
f1_tt

1.0

In [17]:
confusion_matrix(y_test, pred_tt)

array([[12,  0,  0],
       [ 0, 14,  0],
       [ 0,  0, 10]])

부스팅 모델을 이용한 train,test (정확도,f1 score , confusion matrix)확인하기

In [18]:
model_lgb = LGBMClassifier(n_estimators=300, random_state=42)
model_lgb.fit(x_train, y_train)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001144 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 503
[LightGBM] [Info] Number of data points in the train set: 142, number of used features: 13
[LightGBM] [Info] Start training from score -1.105679
[LightGBM] [Info] Start training from score -0.912776
[LightGBM] [Info] Start training from score -1.318241
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,300
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [19]:
pred_l = model_lgb.predict(x_train)
accuracy_score(y_train, pred_l)

c:\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


1.0

In [20]:
pred_lt = model_lgb.predict(x_test)
accuracy_score(y_test, pred_lt)

c:\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


1.0

In [21]:
f1_lgb = f1_score(y_train, pred_l, average='macro')
f1_lgb

1.0

In [22]:
f1_lgbt = f1_score(y_test, pred_lt, average='macro')
f1_lgbt

1.0

In [23]:
cl = confusion_matrix(y_train, pred_l)
cl

array([[47,  0,  0],
       [ 0, 57,  0],
       [ 0,  0, 38]])

In [24]:
clt = confusion_matrix(y_test, pred_lt)
clt

array([[12,  0,  0],
       [ 0, 14,  0],
       [ 0,  0, 10]])

iris 데이터 셋 이용
- 군집화 하기
- 정답과 군집화값 정확도 확인하기 

In [25]:
iris = load_iris(as_frame=True)
iris.data

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2
...,...,...,...,...
145,6.7,3.0,5.2,2.3
146,6.3,2.5,5.0,1.9
147,6.5,3.0,5.2,2.0
148,6.2,3.4,5.4,2.3


In [26]:
iris.target_names

array(['setosa', 'versicolor', 'virginica'], dtype='<U10')

In [27]:
k =  KMeans(n_clusters=3)
km = k.fit(iris.data)

In [28]:
km.cluster_centers_

array([[5.006     , 3.428     , 1.462     , 0.246     ],
       [6.85384615, 3.07692308, 5.71538462, 2.05384615],
       [5.88360656, 2.74098361, 4.38852459, 1.43442623]])

In [29]:
df = iris.data.copy()
df['cluster'] = km.labels_
df['target'] = iris.target

pd.crosstab(df['cluster'], df['target'])

target,0,1,2
cluster,,,
0,50,0,0
1,0,3,36
2,0,47,14


In [30]:
cluster_to_label = (
	df.groupby('cluster')['target']
    .agg(lambda x: x.value_counts().idxmax())
)

cluster_to_label

cluster
0    0
1    2
2    1
Name: target, dtype: int64

In [31]:
y_pred = df['cluster'].map(cluster_to_label)
accuracy_score(df['target'], y_pred)

0.8866666666666667

In [32]:
km.labels_

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 2, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 2, 1, 1, 1, 1, 2, 1, 1, 1,
       1, 1, 1, 2, 2, 1, 1, 1, 1, 2, 1, 2, 1, 2, 1, 1, 2, 2, 1, 1, 1, 1,
       1, 2, 1, 1, 1, 1, 2, 1, 1, 1, 2, 1, 1, 1, 2, 1, 1, 2], dtype=int32)

In [33]:
km.inertia_

78.8556658259773